# Hard rods

In [1]:
import tempfile

import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt
import flax.linen as nn
from matplotlib.colors import LogNorm

from qvarnet.train import train
from qvarnet.config.training_setup import TrainingConfig, ChainInitAndWarmupConfig
from qvarnet.config.coord_mode import LabCoords
from qvarnet.geometry.qgt import QGTConfig
from qvarnet.boundaries import NoBoundary
from qvarnet.models.compose import LogWavefunction
from qvarnet.models.mlp import MLP
from qvarnet.models.envelopes import GaussianEnvelope
from qvarnet.models.jastrow import LogJastrow
from qvarnet.models.analytic import CalogeroSutherlandAnalyticModel
from qvarnet.hamiltonian.continuous import HarmonicOscillatorHamiltonian
from qvarnet.vmc.training_step import compute_local_energy
from qvarnet.callbacks import SnapshotCallback
from qvarnet.estimators import TrainedWavefunction

### We use the harmonic oscillator hamiltonian as the hamiltonian cause the rods are implemented as a condition in the wavefunction.

## Ordered sampler demo

The rod condition (`x1 < x2 < ... < xn`) is enforced entirely by the `"1d-ordered"` sampler (`src/qvarnet/samplers/kernel.py`, `mh_kernel_log_1d_ordered`/`mh_chain_1d_ordered`): every proposal is sorted before the accept/reject test, so the wavefunction below never sees an out-of-order configuration and needs no knowledge of the constraint itself. The (separate, not-yet-implemented) hard-rod condition in the wavefunction — impenetrability at finite rod length — is a placeholder below (`jastrow=None`); swap in whatever term encodes it once decided.

Two things worth keeping in mind when using this sampler:
- No special initialization is needed — `mh_chain_1d_ordered` sorts `init_position` unconditionally on its very first call, so the default warmup (which itself does not enforce ordering) is automatically corrected the moment the `"1d-ordered"` sampler takes over for the main training loop.
- `("particle-subset", {"n_move": 1, "n_dim": 1})` moves one rod at a time — most proposals then only risk crossing one neighbour, keeping the sampler's accept rate reasonable (0.91 in the run below).

In [9]:
n_particles = 5
omega = 1.0
n_chains = 256

hamiltonian = HarmonicOscillatorHamiltonian(omega=omega)

model = LogWavefunction(
    network=MLP(hidden=[32, 32], output_dim=1),
    transform=NoBoundary(),
    envelope=GaussianEnvelope(init=0.5),
    jastrow=None,  # placeholder — the rod condition in the wavefunction goes here
)

sampler_params = {
    "sampler": "1d-ordered",
    "step_size": 0.2,
    "chain_length": 21,
    "thermalization_steps": 20,
    "proposal": ("particle-subset", {"n_move": 1, "n_dim": 1}),
}

with tempfile.TemporaryDirectory() as tmpdir:
    training_config = TrainingConfig(
        n_epochs=2000,
        rng_seed=0,
        checkpoint_path=tmpdir,
    )
    result = train(
        shape=(n_chains, n_particles),
        model=model,
        optimizer=optax.adam(3e-3),
        hamiltonian=hamiltonian,
        training_config=training_config,
        sampler_params=sampler_params,
        coord_mode=LabCoords(),
    )
# training_config.print_summary (default True) already printed the run report above,
# including the best epoch by the "std" selection metric.

# Sanity check: the sampler must never break ordering.
final_positions = np.asarray(result.final_positions)
print(result.__dir__())
still_ordered = np.all(np.diff(final_positions, axis=-1) >= 0)
print(f"All {n_chains} final chains still ordered: {still_ordered}")

  0%|          | 0/2000 [00:00<?, ?it/s]

100%|██████████| 2000/2000 [00:16<00:00, 119.65it/s, E=3.0268, sigma_E=0.0461]


── training summary ────────────────────────────────────────
epochs ran       : 2000   (0m 13.8s wall, 1282 parameters)
final epoch      : E = 3.019559 ± 2.92e-03   σ_E = 0.0467
best epoch ( 1998) : E = 3.018676 ± 2.30e-03   σ_E = 0.0367
acceptance (tail): 0.916
best snapshot    : epoch 1998 (select metric = 0.0367473; 3 kept — result.best_params() to load)
────────────────────────────────────────────────────────────
['history', 'final_params', 'snapshots', 'final_positions', 'final_step_size', '__module__', '__firstlineno__', '__doc__', '__init__', 'best_params', 'best_k', 'best_k_params', 'best_steps', 'cm_mean', 'cm_std', 'best', 'summary', 'diagnose', '__iter__', '__repr__', '__static_attributes__', '__dict__', '__weakref__', '__new__', '__hash__', '__str__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__eq__', '__ne__', '__gt__', '__ge__', '__reduce_ex__', '__reduce__', '__getstate__', '__subclasshook__', '__init_subclass__', '__format__', '__sizeof__', 